# Lab: reliable transport over UDP

TCP gives applications a reliable channel: every byte arrives, once, in order. UDP gives no such guarantee: a datagram
can be lost, and nobody will notice. In this lab you build the missing piece yourself: a **transport protocol that runs
on top of UDP and delivers data reliably over a lossy network**.

The application is as simple as it gets. A client sends a request to a server with a number of bytes, and the server
sends back that many bytes of data. The client measures how long the transfer takes. Between the application and UDP
sits the transport protocol of `apps/protocol.py`. It splits the data into numbered segments, the receiver
acknowledges them, and the sender retransmits what gets lost.

The network is emulated with Mininet and drops a fraction of the packets at random. These losses are **not** caused by
congestion: you do not need to implement congestion control, only to recover from the losses as efficiently as you can.

You are given a working **stop-and-wait** protocol. You will:

1. measure the application without and with loss, and see why plain UDP is not enough and why stop-and-wait is slow;
2. implement three better protocols in `apps/protocol.py`:
   * `window`: Go-Back-N with a window that doubles after every successful round;
   * `sack`: selective repeat with selective acknowledgments (SACK) and a fixed window;
   * `sackwindow`: SACK with the growing window of `window`;
3. compare the four protocols over three loss rates: 0.1%, 1% and 10%.

**What to submit:** this notebook **with all the outputs** (run every cell) and your answers written in it, plus
`apps/protocol.py`.

> **Requirements:** this notebook must run on a Linux machine with Mininet installed, and Jupyter must run as **root**.

## 0. Setup

The network has one switch: `h1` (client, 10.0.0.1) — `s1` — `h2` (server, 10.0.0.2). The `h1 - s1` link has a
bandwidth of `BW` Mbit/s and a delay of `DELAY` **in each direction** (so the RTT is about 10 ms), and drops a fraction
`loss` (in %) of the packets, also **in each direction**. The `h2 - s1` link is not constrained.

The applications are in `apps/`:

* `server.py [--protocol P] [--window W]` waits for requests and sends back the number of bytes asked for;
* `client.py SERVER_IP SIZE [--protocol P] [--window W]` asks for `SIZE` bytes, checks the data it receives and prints
  `RESULT <bytes> <seconds> OK` (or `CORRUPTED`);
* `common.py` holds what the two agree on (the port, the request format, the data the server sends);
* `protocol.py` is the transport protocol, and the only file you need to modify.

You do not need to modify the cells of this section.

In [ ]:
import os
import re
import statistics
import time

from mininet.net import Mininet
from mininet.topo import Topo
from mininet.node import OVSBridge
from mininet.link import TCLink
from mininet.log import setLogLevel
from mininet.clean import cleanup

assert os.geteuid() == 0, "Mininet needs root: start Jupyter with sudo"

setLogLevel("warning")
cleanup()

APPS = os.path.abspath("apps")
SERVER_IP = "10.0.0.2"
SIZE = 1_000_000  # bytes the client asks for
BW = 50           # Mbit/s
DELAY = "5ms"     # in each direction

net = h1 = h2 = None

In [ ]:
class OneSwitchTopo(Topo):
    def build(self, loss=None):
        h1 = self.addHost("h1", ip="10.0.0.1/24")
        h2 = self.addHost("h2", ip="10.0.0.2/24")
        s1 = self.addSwitch("s1")
        self.addLink(h1, s1, bw=BW, delay=DELAY, loss=loss)  # shaped, lossy link
        self.addLink(h2, s1)                                 # unconstrained link


def build_net(loss=None, protocol="stopwait", window=None):
    # (Re)build the network with the given loss (in %) and start the server on h2
    global net, h1, h2
    if net is not None:
        h2.cmd("pkill -f apps/server.py")
        net.stop()
    net = Mininet(topo=OneSwitchTopo(loss=loss), link=TCLink, switch=OVSBridge, controller=None)
    net.start()
    h1, h2 = net.get("h1", "h2")
    h2.cmd(f"python3 -u {APPS}/server.py {options(protocol, window)} > /tmp/server.log 2>&1 &")
    for _ in range(20):
        if ":20003" in h2.cmd("ss -lun"):
            return
        time.sleep(0.25)
    raise RuntimeError("The server did not start:\n" + h2.cmd("cat /tmp/server.log"))


def options(protocol, window):
    return f"--protocol {protocol}" + (f" --window {window}" if window else "")


def transfer(protocol, size=SIZE, window=None, timeout=300, verbose=False):
    # Run the client on h1. Returns the transfer time in seconds, or None if it failed.
    out = h1.cmd(f"timeout {timeout} python3 {APPS}/client.py {SERVER_IP} {size} {options(protocol, window)}")
    if verbose:
        print(out.strip())
    if "NotImplementedError" in out:
        raise NotImplementedError(protocol)
    m = re.search(r"^RESULT (\d+) ([\d.]+) (\w+)", out, re.M)
    if m is None or int(m.group(1)) != size or m.group(3) != "OK":
        print(f"  {protocol}: the transfer failed (or did not finish within {timeout} s):")
        print("  " + "\n  ".join(out.strip().splitlines()[-3:]))
        return None
    return float(m.group(2))


def retransmissions():
    # Retransmissions of the server in its last transfer, in % of the segments
    stats = re.findall(r"^STATS (\d+) (\d+) (\d+)", h2.cmd("cat /tmp/server.log"), re.M)
    if not stats:
        return None
    _, segments, transmissions = map(int, stats[-1])
    return 100 * (transmissions - segments) / segments


def run(protocol, loss=None, reps=1, size=SIZE, window=None):
    # Rebuild the network with the given loss and do `reps` transfers with `protocol`
    build_net(loss=loss, protocol=protocol, window=window)
    times, retx = [], []
    for _ in range(reps):
        t = transfer(protocol, size, window)
        times.append(t)
        if t is not None:
            retx.append(retransmissions())
    print(f"  done: {protocol}, loss {loss or 0}%")
    return {"protocol": protocol, "window": window, "loss": loss, "size": size, "times": times, "retx": retx}


def show(rows):
    print(f"{'protocol':>10} {'window':>6} {'loss %':>6} {'time s (mean ± sd)':>20} {'Mbit/s':>8}"
          f" {'retx %':>7} {'failed':>6}")
    for r in rows:
        ok = [t for t in r["times"] if t is not None]
        failed = len(r["times"]) - len(ok)
        if ok:
            mean = statistics.mean(ok)
            sd = statistics.stdev(ok) if len(ok) > 1 else 0
            time_s = f"{mean:9.3f} ± {sd:8.3f}"
            tput = f"{r['size'] * 8 / mean / 1e6:8.2f}"
            retx = f"{statistics.mean(x for x in r['retx'] if x is not None):7.1f}" if any(
                x is not None for x in r["retx"]) else f"{'-':>7}"
        else:
            time_s, tput, retx = f"{'-':>20}", f"{'-':>8}", f"{'-':>7}"
        print(f"{r['protocol']:>10} {str(r['window'] or '-'):>6} {str(r['loss'] or 0):>6} {time_s:>20} {tput}"
              f" {retx} {failed:>6}")

## 1. Without loss

Read `apps/protocol.py` before going on: the comment at the top describes the format of the segments, and the
`Transport` class does everything except the sender's algorithm. Two protocols are already there:

* `udp` sends every segment once and never retransmits: this is what you get with plain UDP;
* `stopwait` sends one segment, waits for its acknowledgment, and only then sends the next one. If the acknowledgment
  does not arrive within `TIMEOUT` seconds, it retransmits the segment.

Let's transfer `SIZE` bytes with both, on a network without loss. The table shows the transfer time, the throughput
(the size divided by the time), and the segments that the server retransmitted, in % of the segments of the transfer.

In [ ]:
rows = [run("udp"), run("stopwait")]
show(rows)

**Question 1.1** — Both protocols deliver the data correctly. Why is `stopwait` so much slower than `udp`? Compute the
throughput you expect from stop-and-wait using the size of a segment (`MSS`) and the RTT, and compare it with your
measurement.

*Your answer here.*

## 2. With loss

Now the link drops 1% of the packets in each direction.

In [ ]:
rows = [run("udp", loss=1), run("stopwait", loss=1)]
show(rows)

**Question 2.1** — What happens to `udp`, and why? How does `stopwait` recover from a loss, and how much does each loss
cost it in time?

*Your answer here.*

## 3. Your turn: three better protocols

Stop-and-wait is reliable but slow: it leaves the link idle while it waits for each acknowledgment. Implement the
`send_segments()` method of the three classes at the bottom of `apps/protocol.py`. Their docstrings describe the
algorithms step by step.

| protocol | class | window | retransmissions |
|---|---|---|---|
| `window` | `GrowingWindow` | starts at 1 segment, doubles after every round acknowledged without a timeout, back to 1 after a timeout | Go-Back-N: on a timeout, everything from the oldest unacknowledged segment |
| `sack` | `SelectiveAck` | fixed, `--window` segments (32 by default) | only the missing segments, using the SACK blocks of the ACKs |
| `sackwindow` | `GrowingSack` | as `window` | as `sack` |

A few tips:

* Use `self.transmit(seg)` to send a segment and `self.wait_for_ack(deadline)` to wait for the next ACK. Do not use
  the socket directly. The receiver side is already implemented and works with every protocol.
* `ack.ack` is the cumulative ACK, and `ack.acknowledges(seq)` tells you whether a segment has been received,
  including the SACK blocks.
* The client uses the same protocol as the server to send its request, so your code runs on both sides.
* The server prints errors in `/tmp/server.log`: `print(h2.cmd("cat /tmp/server.log"))`.

Every time you change `protocol.py`, rerun the cell below to test one protocol. `run()` rebuilds the network, so
the server restarts with your new code. Start with small transfers and no loss, then add loss.

In [ ]:
show([run("window", loss=1, size=100_000)])

## 4. Performance over three loss rates

Finally, compare the four protocols with 0.1%, 1% and 10% loss, with 3 transfers each. Protocols that are not
implemented yet are skipped. This takes a few minutes: `stopwait` with 10% loss is slow.

In [ ]:
PROTOCOLS = ["stopwait", "window", "sack", "sackwindow"]
LOSSES = [0.1, 1, 10]

results = []
for protocol in PROTOCOLS:
    for loss in LOSSES:
        try:
            results.append(run(protocol, loss=loss, reps=3))
        except NotImplementedError:
            print(f"  {protocol} is not implemented yet: skipped")
            break
show(results)

## 5. Questions

Answer using **your** measurements: quote numbers from your tables.

**Question 5.1** — Rank the four protocols at each loss rate. Which one is the best at 0.1%, and which at 10%? Is the
ranking the same at every loss rate? Why?

*Your answer here.*

**Question 5.2** — Compare the retransmissions (`retx %`) of `window` and `sack` at 10% loss. Why does Go-Back-N
retransmit so much more than the loss rate? Which segments does it retransmit needlessly?

*Your answer here.*

**Question 5.3** — `window` and `sackwindow` go back to a window of 1 segment after every timeout. This is what TCP does
on a timeout, because it assumes that the loss was caused by congestion. Is it a good idea here, where losses are
random? What would you do instead, and what could go wrong if the losses were in fact caused by congestion?

*Your answer here.*

**Question 5.4** — With `sack`, which window size do you need to use the whole bandwidth of the link? Relate it to the
bandwidth-delay product. Try a few values with `run("sack", loss=1, reps=3, window=W)` and report the results.

*Your answer here.*

**Question 5.5** — `TIMEOUT` is fixed to 50 ms. What would happen with an RTT of 200 ms? And with an RTT of 1 ms? How
does TCP choose its retransmission timeout?

*Your answer here.*

In [ ]:
# Clean up when you are done
h2.cmd("pkill -f apps/server.py")
net.stop()

# Submission instructions

Please upload an HTML export of this notebook, as well as the complete code, to the lab's submission page. You should
submit your work within one hour of the end of the lab. Late submissions will be graded according to the course's
policy.

Before submission, run the following command from a terminal:

```bash
jupyter nbconvert --to html assignment.ipynb
```

If the command fails, you might have to install nbconvert:

```bash
sudo apt install jupyter-nbconvert
```

Then compress the folder:

```bash
tar -czvf submission.tar.gz reliable_transport/
```

And upload it to the submission page.